In [4]:
# %% [Part 0] 写配置文件 —— 只需运行一次，改了参数才需要重新运行这个cell
from pathlib import Path

PROJECT_ROOT = Path("/Volumes/Elements/DEPA_downstream")
yaml_path = PROJECT_ROOT / "configs" / "pre_t.yaml"

yaml_content = """cnmf_dir: "/Volumes/Elements/data/ALL_Pre-T/cNMF_output"
run_name: "PreT_DEAPA_programs"
K: 8
density_threshold: "0_01"

h5ad_path: "/Volumes/Elements/data/ALL_Pre-T/cNMF_output/PreT_DEAPA_counts.h5ad"
feature_anno_path: "/Volumes/Elements/data/ALL_Pre-T/cNMF_input/cnmf_feature_anno.csv"

wpsi_matrix_path: "/Volumes/Elements/data/ALL_Pre-T/PSI_results/movAPA_wPSI_matrix_df.rds"
psi_matrix_path: "/Volumes/Elements/data/ALL_Pre-T/PSI_results/movAPA_PSI_matrix_df.rds"
depa_feature_source_path: "/Volumes/Elements/data/ALL_Pre-T/cNMF_input/DEPA_feature_source.csv"
gene_symbol_map_path: "/Volumes/Elements/DEPA_downstream/configs/pre_t_gene_symbol_map.csv"

meta_columns:
  sample: "sample"
  celltype: "cell_type_final"
  umap1: "UMAP_1"
  umap2: "UMAP_2"

contrasts:
  - name: "disease"
    column: "group"
    type: "pairwise"
    group_a: "PBMMC"
    group_b: "PreT_ALL"
    category: "disease"

  - name: "celltype_identity"
    column: "cell_type_final"
    type: "auto_one_vs_rest_all"

z_threshold: 1.5
background_scope: "pac_universe"
min_cells_per_group: 20
enrichr_databases: [GO_Biological_Process_2023, KEGG_2021_Human, MSigDB_Hallmark_2020, Reactome_2022]
output_root: "/Volumes/Elements/DEPA_downstream/outputs/pre_T/"
"""
yaml_path.write_text(yaml_content, encoding="utf-8")
print(f"写入完成: {yaml_path}")

# 加个保护：文件已存在就不覆盖，改配置时手动删掉旧文件或直接编辑yaml本身
if yaml_path.exists():
    print(f"配置文件已存在，跳过写入: {yaml_path}\n"
          f"（如需更新配置，直接编辑该yaml文件，或删除后重新运行本cell）")
else:
    yaml_path.write_text(yaml_content, encoding="utf-8")
    print(f"写入完成: {yaml_path}")

写入完成: /Volumes/Elements/DEPA_downstream/configs/pre_t.yaml
配置文件已存在，跳过写入: /Volumes/Elements/DEPA_downstream/configs/pre_t.yaml
（如需更新配置，直接编辑该yaml文件，或删除后重新运行本cell）


In [5]:
# %% [Part 0] 加载依赖与数据 —— 每次重启kernel后运行一次即可，
# 后面三层分析都复用这里的 data / CONFIG，不用重复加载
%load_ext autoreload
%autoreload 2

import yaml
import pandas as pd
from pathlib import Path
from depa_downstream import (
    load_cnmf_outputs,
    compute_program_direction, plot_program_direction_summary,
    compute_leading_genes, compare_within_celltype, summarize_leading_gene_within_celltype,   # 新增后两个
    compute_contrast_association, plot_association_heatmap, plot_program_umap_grid,
    compute_program_gene_sets, compute_jaccard_matrix, select_program_markers,
    resolve_module_labels, plot_jaccard_heatmap,
    run_enrichment, plot_enrichment_bubble,
    plot_leading_genes_bar, plot_gene_wpsi_by_group,           # 之前加过的也顺带确认在里面
    plot_within_celltype_comparison, plot_within_celltype_effect_summary,   # 这次新增
)

CONFIG = yaml.safe_load(open(PROJECT_ROOT / "configs" / "pre_t.yaml"))
Path(CONFIG["output_root"]).mkdir(parents=True, exist_ok=True)
print("gene_symbol_map_path:", CONFIG["gene_symbol_map_path"])
CONFIG

data = load_cnmf_outputs(CONFIG)
print("programs:", data["programs"])
print("meta细胞数:", len(data["meta"]))
print("cell_type_final取值:", data["meta"]["cell_type_final"].unique())


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
gene_symbol_map_path: /Volumes/Elements/DEPA_downstream/configs/pre_t_gene_symbol_map.csv
检查 cNMF 输入文件...
  [OK] spectra_score: /Volumes/Elements/data/ALL_Pre-T/cNMF_output/PreT_DEAPA_programs/PreT_DEAPA_programs.gene_spectra_score.k_8.dt_0_01.txt (0.1 MB)
  [OK] spectra_tpm: /Volumes/Elements/data/ALL_Pre-T/cNMF_output/PreT_DEAPA_programs/PreT_DEAPA_programs.gene_spectra_tpm.k_8.dt_0_01.txt (0.0 MB)
  [OK] usage: /Volumes/Elements/data/ALL_Pre-T/cNMF_output/PreT_DEAPA_programs/PreT_DEAPA_programs.usages.k_8.dt_0_01.consensus.txt (1.6 MB)

读取 cNMF spectra 与 usage...
  spectra: (8, 645)  (programs x PACs)
  usage:   (17387, 8)  (cells x programs)
  programs (已统一命名): ['P01', 'P02', 'P03', 'P04', 'P05', 'P06', 'P07', 'P08']

计算 spectra 逐 program 的 Z-score...
  spectra_z: (8, 645)

读取 metadata (h5ad .obs)...
  meta: (17387, 8)  (cells x columns)
  sample     -> 列 'sample'，取值样例: ['PreT_01', 'PreT_02', 'P

In [6]:
# %% [Layer 1] program 的 wPSI 方向（Proximal / Distal）
program_direction = compute_program_direction(data, CONFIG)
plot_program_direction_summary(program_direction, CONFIG, CONFIG["output_root"] + "layer1")
program_direction


计算基因级 wPSI（7788 个基因 x 17387 个细胞）...
  基因检出率过滤(min_detection_rate=0.1): 排除 3084/7788 个基因(wPSI在多数细胞中卡在0/1边界，中位数不可靠)
  过滤后有效基因数: 4704
  cNMF分析涉及的基因全集（pac_universe，基于645个PAC）大小: 393
  background_scope = 'pac_universe'
  [P01] 前景基因=23, 背景基因=370, delta_median=-0.0898, pval=2.98e-01 -> Proximal
  [P02] 前景基因=34, 背景基因=359, delta_median=-0.1518, pval=1.30e-02 -> Proximal
  [P03] 前景基因=20, 背景基因=373, delta_median=-0.1059, pval=9.37e-01 -> Proximal
  [P04] 前景基因=25, 背景基因=368, delta_median=+0.0850, pval=8.49e-01 -> Distal
  [P05] 前景基因=33, 背景基因=360, delta_median=-0.1446, pval=2.69e-01 -> Proximal
  [P06] 前景基因=14, 背景基因=379, delta_median=-0.2298, pval=4.21e-01 -> Proximal
  [P07] 前景基因=16, 背景基因=377, delta_median=+0.0304, pval=8.28e-01 -> Distal
  [P08] 前景基因=16, 背景基因=377, delta_median=-0.1740, pval=2.89e-01 -> Proximal

共 8/8 个 program 完成方向检验
  已保存: /Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_program_direction_summary.png


,n_fg_genes,n_bg_genes,fg_median,bg_median,delta_median,mwu_stat,pval,direction,padj
program,,,,,,,,,
P01,23,370,0.234725,0.324556,-0.089831,3705.5,0.298154,Proximal,0.596307
P02,34,359,0.176873,0.328675,-0.151802,4532.0,0.012951,Proximal,0.103611
P03,20,373,0.213374,0.319286,-0.105912,3690.5,0.937082,Proximal,0.937082
P04,25,368,0.394930,0.309949,0.084982,4495.0,0.848945,Distal,0.937082
P05,33,360,0.179988,0.324556,-0.144568,5250.5,0.269113,Proximal,0.596307
P06,14,379,0.089507,0.319286,-0.229779,2317.0,0.420706,Proximal,0.673130
P07,16,377,0.340041,0.309691,0.030350,2919.0,0.828041,Distal,0.937082
P08,16,377,0.148838,0.322804,-0.173966,2544.0,0.288547,Proximal,0.596307


In [7]:
# %% [Layer 1 深入] leading genes：谁在驱动每个program的方向
leading_genes = compute_leading_genes(data, CONFIG, program_direction, top_n=15)
leading_genes["P02"]


  基因检出率过滤(min_detection_rate=0.1): 排除 3084/7788 个基因(wPSI在多数细胞中卡在0/1边界，中位数不可靠)
  过滤后有效基因数: 4704
  [P01] direction=Proximal, leading genes: ['RSRP1', 'EIF3E', 'RPL34', 'RPL13', 'SRSF5']...
  [P02] direction=Proximal, leading genes: ['CENPM', 'PSMB2', 'CARHSP1', 'IDH2', 'SUB1']...
  [P03] direction=Proximal, leading genes: ['TAGAP', 'HRK', 'CYTIP', 'DEK', 'CD74']...
  [P04] direction=Distal, leading genes: ['CXCL8', 'IRAK3', 'MEGF9', 'MAFB', 'FGR']...
  [P05] direction=Proximal, leading genes: ['GLRX3', 'CENPU', 'CENPM', 'LMNA', 'NUCLEOLIN']...
  [P06] direction=Proximal, leading genes: ['GYPE', 'RNA5SP78', 'HBB', 'HBD', 'TESC']...
  [P07] direction=Distal, leading genes: ['UFM1', 'KDELR2', 'SRPRB', 'MESD', 'TMBIM6']...
  [P08] direction=Proximal, leading genes: ['GYPE', 'LMNA', 'HBB', 'HBD', 'MBOAT2']...


,gene_id,symbol,gene_wpsi,delta_from_bg,spectra_z_max
0,ENSG00000100162.15,CENPM,0.000000,-0.328675,2.656970
1,ENSG00000126067.12,PSMB2,0.000000,-0.328675,2.242298
2,ENSG00000153048.11,CARHSP1,0.000000,-0.328675,2.621932
3,ENSG00000182054.10,IDH2,0.000000,-0.328675,4.090217
4,ENSG00000113387.12,SUB1,0.000000,-0.328675,2.417758
5,ENSG00000151725.12,CENPU,0.000000,-0.328675,1.549971
6,ENSG00000177733.7,HNRNPA0,0.000457,-0.328218,4.009486
7,ENSG00000171848.16,RRM2,0.003243,-0.325432,3.827629
8,ENSG00000116161.18,CACYBP,0.008343,-0.320332,2.757901
9,ENSG00000107447.8,DNTT,0.027542,-0.301134,3.265012


In [8]:
# %% [Layer 1 深入] leading genes条形图（对所有完成检验的program批量画）
for prog in data["programs"]:
    direction = program_direction.loc[prog, "direction"]
    if direction != "NA":
        plot_leading_genes_bar(leading_genes, prog, CONFIG["output_root"] + "layer1", direction_label=direction)

  已保存: /Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_leading_genes_P01.png
  已保存: /Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_leading_genes_P02.png
  已保存: /Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_leading_genes_P03.png
  已保存: /Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_leading_genes_P04.png
  已保存: /Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_leading_genes_P05.png
  已保存: /Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_leading_genes_P06.png
  已保存: /Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_leading_genes_P07.png
  已保存: /Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_leading_genes_P08.png


In [9]:
# %% [Layer 1 深入] 挑P02的top leading gene，看它在disease分组下的具体wPSI分布
top_gene = leading_genes["P02"].iloc[0]
plot_gene_wpsi_by_group(
    top_gene["gene_id"], data, CONFIG,
    groupby_col=CONFIG["contrasts"][0]["column"],  # "group" -> PBMMC/PreT_ALL
    out_dir=CONFIG["output_root"] + "layer1",
    gene_symbol=top_gene["symbol"],
)

  已保存: /Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_gene_wpsi_CENPM_group.png


'/Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_gene_wpsi_CENPM_group.png'

In [10]:
# %% [Layer 1 深入] 同一个基因也可以按细胞类型看
plot_gene_wpsi_by_group(
    top_gene["gene_id"], data, CONFIG,
    groupby_col=CONFIG["meta_columns"]["celltype"],
    out_dir=CONFIG["output_root"] + "layer1",
    gene_symbol=top_gene["symbol"],
)

  已保存: /Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_gene_wpsi_CENPM_cell_type_final.png


'/Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_gene_wpsi_CENPM_cell_type_final.png'

In [11]:
# %% [Layer 1 深入] 组内比较：限定在 Proliferating T-ALL 内部，
# 比较PBMMC来源(正常增殖细胞) vs PreT_ALL来源(恶性增殖细胞)
within_result = compare_within_celltype(
    data, CONFIG,
    celltype="Proliferating T-ALL",
    disease_column=CONFIG["contrasts"][0]["column"],   # "group"
    group_a="PBMMC", group_b="PreT_ALL",
    gene_id=top_gene["gene_id"],   # 复用之前leading gene里选的CENPM
    program="P02",
)
within_result

=== 组内比较: celltype='Proliferating T-ALL', group: PBMMC vs PreT_ALL ===
  该celltype总细胞数: 3020 (PBMMC: 733, PreT_ALL: 2287)
  [wPSI] PBMMC median=0.0000 (n=324)  PreT_ALL median=0.0000 (n=978)  pval=2.610e-07
  [usage:P02] PBMMC median=0.2747 (n=733)  PreT_ALL median=0.6281 (n=2287)  pval=1.183e-257


{'wpsi': {'n_a': 324,
  'n_b': 978,
  'median_a': 0.0,
  'median_b': 0.0,
  'pval': 2.609719203543101e-07},
 'usage': {'n_a': 733,
  'n_b': 2287,
  'median_a': 0.2747449,
  'median_b': 0.62805843,
  'pval': 1.1827166820433264e-257}}

In [12]:
# %% [Layer 1 深入] 组内比较画图 —— wPSI层面
wpsi = data["wpsi"]
meta = data["meta"]
celltype_col = CONFIG["meta_columns"]["celltype"]
disease_col = CONFIG["contrasts"][0]["column"]

mask = meta[celltype_col] == "Proliferating T-ALL"
sub_meta = meta.loc[mask]
common_cells = wpsi.columns.intersection(sub_meta.index)
vals = wpsi.loc[top_gene["gene_id"], common_cells]
groups = sub_meta.loc[common_cells, disease_col]

plot_within_celltype_comparison(
    vals[groups == "PBMMC"].dropna().values,
    vals[groups == "PreT_ALL"].dropna().values,
    label_a="PBMMC\n(normal proliferating)",
    label_b="PreT_ALL\n(malignant proliferating)",
    ylabel="wPSI (per cell)",
    title=f"{top_gene['symbol']} wPSI — within Proliferating T-ALL cluster",
    out_path=CONFIG["output_root"] + f"layer1/layer1_within_celltype_wpsi_{top_gene['symbol']}.png",
)

  已保存: /Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_within_celltype_wpsi_CENPM.png


'/Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_within_celltype_wpsi_CENPM.png'

In [13]:
# %% [Layer 1 深入] 组内比较画图 —— P02 usage层面
usage = data["usage"]
common_cells = usage.index.intersection(sub_meta.index)
u_vals = usage.loc[common_cells, "P02"]
u_groups = sub_meta.loc[common_cells, disease_col]

plot_within_celltype_comparison(
    u_vals[u_groups == "PBMMC"].dropna().values,
    u_vals[u_groups == "PreT_ALL"].dropna().values,
    label_a="PBMMC\n(normal proliferating)",
    label_b="PreT_ALL\n(malignant proliferating)",
    ylabel="P02 usage",
    title="P02 usage — within Proliferating T-ALL cluster",
    out_path=CONFIG["output_root"] + "layer1/layer1_within_celltype_usage_P02.png",
)

  已保存: /Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_within_celltype_usage_P02.png


'/Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_within_celltype_usage_P02.png'

In [14]:
# 1. 看两组的检出率是不是本身就不一样
print("PBMMC检出率:", 324/733, " PreT_ALL检出率:", 978/2287)

# 2. 效应量：非零细胞比例，比中位数更能反映真实差异在哪
a_vals = vals[groups == "PBMMC"].dropna()
b_vals = vals[groups == "PreT_ALL"].dropna()
print("PBMMC中 wPSI>0 的比例:", (a_vals > 0).mean())
print("PreT_ALL中 wPSI>0 的比例:", (b_vals > 0).mean())
print("PBMMC均值:", a_vals.mean(), " PreT_ALL均值:", b_vals.mean())

PBMMC检出率: 0.44201909959072305  PreT_ALL检出率: 0.4276344556187145
PBMMC中 wPSI>0 的比例: 0.1388888888888889
PreT_ALL中 wPSI>0 的比例: 0.26482617586912066
PBMMC均值: 0.012903929898785865  PreT_ALL均值: 0.06838871854243361


In [15]:
# 系统性跑一遍P02所有leading gene的组内对照
summary_rows = []
for _, row in leading_genes["P02"].iterrows():
    res = compare_within_celltype(
        data, CONFIG,
        celltype="Proliferating T-ALL",
        disease_column=CONFIG["contrasts"][0]["column"],
        group_a="PBMMC", group_b="PreT_ALL",
        gene_id=row["gene_id"],
    )
    if "wpsi" in res:
        summary_rows.append({
            "symbol": row["symbol"],
            "median_a": res["wpsi"]["median_a"],
            "median_b": res["wpsi"]["median_b"],
            "pval": res["wpsi"]["pval"],
        })

summary_df = pd.DataFrame(summary_rows)
summary_df["direction_consistent"] = summary_df["median_b"] > summary_df["median_a"]  # 是否和P02.usage同方向
summary_df

=== 组内比较: celltype='Proliferating T-ALL', group: PBMMC vs PreT_ALL ===
  该celltype总细胞数: 3020 (PBMMC: 733, PreT_ALL: 2287)
  [wPSI] PBMMC median=0.0000 (n=324)  PreT_ALL median=0.0000 (n=978)  pval=2.610e-07
=== 组内比较: celltype='Proliferating T-ALL', group: PBMMC vs PreT_ALL ===
  该celltype总细胞数: 3020 (PBMMC: 733, PreT_ALL: 2287)
  [wPSI] PBMMC median=0.0000 (n=355)  PreT_ALL median=0.0000 (n=1592)  pval=1.448e-04
=== 组内比较: celltype='Proliferating T-ALL', group: PBMMC vs PreT_ALL ===
  该celltype总细胞数: 3020 (PBMMC: 733, PreT_ALL: 2287)
  [wPSI] PBMMC median=0.0000 (n=439)  PreT_ALL median=0.0000 (n=1308)  pval=1.206e-02
=== 组内比较: celltype='Proliferating T-ALL', group: PBMMC vs PreT_ALL ===
  该celltype总细胞数: 3020 (PBMMC: 733, PreT_ALL: 2287)
  [wPSI] PBMMC median=0.0000 (n=441)  PreT_ALL median=0.0000 (n=1858)  pval=1.510e-01
=== 组内比较: celltype='Proliferating T-ALL', group: PBMMC vs PreT_ALL ===
  该celltype总细胞数: 3020 (PBMMC: 733, PreT_ALL: 2287)
  [wPSI] PBMMC median=0.0015 (n=630)  PreT_ALL 

,symbol,median_a,median_b,pval,direction_consistent
0,CENPM,0.000000,0.000000,2.609719e-07,False
1,PSMB2,0.000000,0.000000,1.447945e-04,False
2,CARHSP1,0.000000,0.000000,1.206250e-02,False
3,IDH2,0.000000,0.000000,1.510429e-01,False
4,SUB1,0.001453,0.000000,1.722339e-11,False
5,CENPU,0.000000,0.000000,5.817162e-02,False
6,HNRNPA0,0.000457,0.034719,1.722766e-08,True
7,RRM2,0.009736,0.003237,7.591332e-41,False
8,CACYBP,0.124811,0.000000,1.437872e-07,False
9,DNTT,0.017821,0.027542,3.145596e-21,True


In [16]:
def compare_within_celltype_effect_size(gene_id, data, config, celltype, disease_column, group_a, group_b):
    """用非零比例+均值代替median比较，避免稀疏性导致的误判（同CENPM诊断逻辑）"""
    wpsi = data["wpsi"]
    meta = data["meta"]
    celltype_col = config["meta_columns"]["celltype"]
    mask = meta[celltype_col] == celltype
    sub_meta = meta.loc[mask]
    common_cells = wpsi.columns.intersection(sub_meta.index)
    vals = wpsi.loc[gene_id, common_cells]
    groups = sub_meta.loc[common_cells, disease_column]
    a_vals = vals[groups == group_a].dropna()
    b_vals = vals[groups == group_b].dropna()
    if len(a_vals) < 20 or len(b_vals) < 20:
        return None
    from scipy.stats import mannwhitneyu
    _, pval = mannwhitneyu(a_vals, b_vals, alternative="two-sided")
    return {
        "nonzero_frac_a": (a_vals > 0).mean(), "nonzero_frac_b": (b_vals > 0).mean(),
        "mean_a": a_vals.mean(), "mean_b": b_vals.mean(),
        "pval": pval,
    }

rows = []
for _, row in leading_genes["P02"].iterrows():
    res = compare_within_celltype_effect_size(
        row["gene_id"], data, CONFIG,
        celltype="Proliferating T-ALL",
        disease_column=CONFIG["contrasts"][0]["column"],
        group_a="PBMMC", group_b="PreT_ALL",
    )
    if res:
        rows.append({"symbol": row["symbol"], **res})

effect_df = pd.DataFrame(rows)
effect_df["mean_diff"] = effect_df["mean_b"] - effect_df["mean_a"]
effect_df["direction_consistent"] = effect_df["mean_diff"] > 0
effect_df["significant"] = effect_df["pval"] < 0.05
effect_df.sort_values("pval")

,symbol,nonzero_frac_a,nonzero_frac_b,mean_a,mean_b,pval,mean_diff,direction_consistent,significant
12,NAP1L1,0.924071,0.970120,0.103491,0.174199,5.486912e-92,0.070707,True,True
7,RRM2,1.000000,0.980469,0.027468,0.024979,7.591332e-41,-0.002489,False,True
9,DNTT,0.980296,0.995673,0.021074,0.035938,3.145596e-21,0.014864,True,True
10,RAN,1.000000,0.998999,0.099931,0.110764,5.019539e-15,0.010833,True,True
11,HMGB1,1.000000,1.000000,0.114530,0.105714,1.058064e-12,-0.008816,False,True
4,SUB1,0.700000,0.413120,0.054961,0.048471,1.722339e-11,-0.006490,False,True
6,HNRNPA0,0.650000,1.000000,0.100072,0.078157,1.722766e-08,-0.021915,False,True
8,CACYBP,0.765714,0.484701,0.210095,0.190650,1.437872e-07,-0.019444,False,True
0,CENPM,0.138889,0.264826,0.012904,0.068389,2.609719e-07,0.055485,True,True
14,RPL15,0.695955,0.884882,0.117161,0.126936,2.172165e-06,0.009776,True,True


In [17]:
effect_df["abs_mean_diff"] = effect_df["mean_diff"].abs()
effect_df_sorted = effect_df.sort_values("abs_mean_diff", ascending=False)
effect_df_sorted

,symbol,nonzero_frac_a,nonzero_frac_b,mean_a,mean_b,pval,mean_diff,direction_consistent,significant,abs_mean_diff
12,NAP1L1,0.924071,0.970120,0.103491,0.174199,5.486912e-92,0.070707,True,True,0.070707
0,CENPM,0.138889,0.264826,0.012904,0.068389,2.609719e-07,0.055485,True,True,0.055485
13,PSIP1,0.893750,0.883968,0.152546,0.125015,1.999849e-01,-0.027531,False,False,0.027531
6,HNRNPA0,0.650000,1.000000,0.100072,0.078157,1.722766e-08,-0.021915,False,True,0.021915
8,CACYBP,0.765714,0.484701,0.210095,0.190650,1.437872e-07,-0.019444,False,True,0.019444
2,CARHSP1,0.252847,0.308869,0.098857,0.115268,1.206250e-02,0.016411,True,True,0.016411
1,PSMB2,0.163380,0.261307,0.038408,0.054547,1.447945e-04,0.016139,True,True,0.016139
9,DNTT,0.980296,0.995673,0.021074,0.035938,3.145596e-21,0.014864,True,True,0.014864
5,CENPU,0.364641,0.292683,0.203279,0.191433,5.817162e-02,-0.011846,False,False,0.011846
10,RAN,1.000000,0.998999,0.099931,0.110764,5.019539e-15,0.010833,True,True,0.010833


In [18]:
pd.set_option("display.max_rows", 20)
effect_df_sorted

,symbol,nonzero_frac_a,nonzero_frac_b,mean_a,mean_b,pval,mean_diff,direction_consistent,significant,abs_mean_diff
12,NAP1L1,0.924071,0.970120,0.103491,0.174199,5.486912e-92,0.070707,True,True,0.070707
0,CENPM,0.138889,0.264826,0.012904,0.068389,2.609719e-07,0.055485,True,True,0.055485
13,PSIP1,0.893750,0.883968,0.152546,0.125015,1.999849e-01,-0.027531,False,False,0.027531
6,HNRNPA0,0.650000,1.000000,0.100072,0.078157,1.722766e-08,-0.021915,False,True,0.021915
8,CACYBP,0.765714,0.484701,0.210095,0.190650,1.437872e-07,-0.019444,False,True,0.019444
2,CARHSP1,0.252847,0.308869,0.098857,0.115268,1.206250e-02,0.016411,True,True,0.016411
1,PSMB2,0.163380,0.261307,0.038408,0.054547,1.447945e-04,0.016139,True,True,0.016139
9,DNTT,0.980296,0.995673,0.021074,0.035938,3.145596e-21,0.014864,True,True,0.014864
5,CENPU,0.364641,0.292683,0.203279,0.191433,5.817162e-02,-0.011846,False,False,0.011846
10,RAN,1.000000,0.998999,0.099931,0.110764,5.019539e-15,0.010833,True,True,0.010833


In [19]:
threshold = 0.01
def classify(row):
    if row["pval"] >= 0.05 or row["abs_mean_diff"] < threshold:
        return "no_meaningful_diff"
    return "supports_P02" if row["mean_diff"] > 0 else "opposes_P02"

effect_df_sorted["category"] = effect_df_sorted.apply(classify, axis=1)
effect_df_sorted["category"].value_counts()

category
no_meaningful_diff    7
supports_P02          6
opposes_P02           2
Name: count, dtype: int64

In [20]:
meaningful = effect_df_sorted[effect_df_sorted["category"] != "no_meaningful_diff"].sort_values("mean_diff", ascending=False)
meaningful

,symbol,nonzero_frac_a,nonzero_frac_b,mean_a,mean_b,pval,mean_diff,direction_consistent,significant,abs_mean_diff,category
12,NAP1L1,0.924071,0.970120,0.103491,0.174199,5.486912e-92,0.070707,True,True,0.070707,supports_P02
0,CENPM,0.138889,0.264826,0.012904,0.068389,2.609719e-07,0.055485,True,True,0.055485,supports_P02
2,CARHSP1,0.252847,0.308869,0.098857,0.115268,1.206250e-02,0.016411,True,True,0.016411,supports_P02
1,PSMB2,0.163380,0.261307,0.038408,0.054547,1.447945e-04,0.016139,True,True,0.016139,supports_P02
9,DNTT,0.980296,0.995673,0.021074,0.035938,3.145596e-21,0.014864,True,True,0.014864,supports_P02
10,RAN,1.000000,0.998999,0.099931,0.110764,5.019539e-15,0.010833,True,True,0.010833,supports_P02
8,CACYBP,0.765714,0.484701,0.210095,0.190650,1.437872e-07,-0.019444,False,True,0.019444,opposes_P02
6,HNRNPA0,0.650000,1.000000,0.100072,0.078157,1.722766e-08,-0.021915,False,True,0.021915,opposes_P02


In [21]:
# %% [Layer 1 深入] P02 leading gene 组内对照效应量汇总 + 分类
effect_df_P02 = summarize_leading_gene_within_celltype(
    "P02", leading_genes, data, CONFIG,
    celltype="Proliferating T-ALL",
    disease_column=CONFIG["contrasts"][0]["column"],
    group_a="PBMMC", group_b="PreT_ALL",
)
effect_df_P02

  [P02] leading gene组内对照分类: supports=6, opposes=2, no_meaningful_diff=7


,gene_id,nonzero_frac_a,nonzero_frac_b,mean_a,mean_b,pval,mean_diff,abs_mean_diff,category
symbol,,,,,,,,,
NAP1L1,ENSG00000187109.15,0.924071,0.970120,0.103491,0.174199,5.486912e-92,0.070707,0.070707,supports
CENPM,ENSG00000100162.15,0.138889,0.264826,0.012904,0.068389,2.609719e-07,0.055485,0.055485,supports
PSIP1,ENSG00000164985.15,0.893750,0.883968,0.152546,0.125015,1.999849e-01,-0.027531,0.027531,no_meaningful_diff
HNRNPA0,ENSG00000177733.7,0.650000,1.000000,0.100072,0.078157,1.722766e-08,-0.021915,0.021915,opposes
CACYBP,ENSG00000116161.18,0.765714,0.484701,0.210095,0.190650,1.437872e-07,-0.019444,0.019444,opposes
CARHSP1,ENSG00000153048.11,0.252847,0.308869,0.098857,0.115268,1.206250e-02,0.016411,0.016411,supports
PSMB2,ENSG00000126067.12,0.163380,0.261307,0.038408,0.054547,1.447945e-04,0.016139,0.016139,supports
DNTT,ENSG00000107447.8,0.980296,0.995673,0.021074,0.035938,3.145596e-21,0.014864,0.014864,supports
CENPU,ENSG00000151725.12,0.364641,0.292683,0.203279,0.191433,5.817162e-02,-0.011846,0.011846,no_meaningful_diff


In [22]:
# %% [Layer 1 深入] 画图
plot_within_celltype_effect_summary(
    effect_df_P02, "P02", "Proliferating T-ALL",
    CONFIG["output_root"] + "layer1",
)

  已保存: /Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_within_celltype_effect_P02.png


'/Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_within_celltype_effect_P02.png'

In [23]:
# %% [Layer 1 深入] P04 leading gene 组内对照效应量汇总 + 分类
# P04 对 disease 的 AUC 只有0.33，对 Monocytes 的 AUC 高达0.99 ——
# 这里验证的是"肿瘤微环境对正常单核细胞的旁观者效应"，不是排除疾病混淆
effect_df_P04 = summarize_leading_gene_within_celltype(
    "P04", leading_genes, data, CONFIG,
    celltype="Monocytes",
    disease_column=CONFIG["contrasts"][0]["column"],
    group_a="PBMMC", group_b="PreT_ALL",
)
effect_df_P04

  [P04] leading gene组内对照分类: supports=5, opposes=3, no_meaningful_diff=6


,gene_id,nonzero_frac_a,nonzero_frac_b,mean_a,mean_b,pval,mean_diff,abs_mean_diff,category
symbol,,,,,,,,,
TMEM107,ENSG00000179029.15,0.984869,1.000000,0.527117,0.302145,6.635785e-47,-0.224972,0.224972,opposes
IRAK3,ENSG00000090376.11,0.846154,1.000000,0.782020,1.000000,4.108406e-06,0.217980,0.217980,supports
KLF6,ENSG00000067082.15,0.995620,1.000000,0.112573,0.298145,5.343476e-105,0.185573,0.185573,supports
MEGF9,ENSG00000106780.9,0.873016,1.000000,0.845533,0.997689,4.504536e-09,0.152155,0.152155,supports
AP1S2,ENSG00000182287.15,0.990476,1.000000,0.338789,0.409278,8.242429e-10,0.070489,0.070489,supports
FTH1,ENSG00000167996.16,1.000000,1.000000,0.432365,0.500872,2.128232e-21,0.068507,0.068507,supports
CXCL8,ENSG00000169429.12,0.998592,1.000000,0.938820,0.876715,3.675499e-10,-0.062105,0.062105,opposes
VCAN,ENSG00000038427.16,0.735426,0.613445,0.456003,0.396429,3.731458e-01,-0.059575,0.059575,no_meaningful_diff
MS4A6A,ENSG00000110077.15,0.932320,0.928144,0.495685,0.449800,9.573844e-02,-0.045885,0.045885,no_meaningful_diff


In [24]:
# %% [Layer 1 深入] P04 画图
plot_within_celltype_effect_summary(
    effect_df_P04, "P04", "Monocytes",
    CONFIG["output_root"] + "layer1",
)

  已保存: /Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_within_celltype_effect_P04.png


'/Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_within_celltype_effect_P04.png'

In [25]:
# %% [Layer 1 深入] P07 leading gene 组内对照效应量汇总 + 分类
# P07 对 disease 的 AUC 只有0.56，对 Plasma cells 的 AUC 高达0.98
effect_df_P07 = summarize_leading_gene_within_celltype(
    "P07", leading_genes, data, CONFIG,
    celltype="Plasma cells",
    disease_column=CONFIG["contrasts"][0]["column"],
    group_a="PBMMC", group_b="PreT_ALL",
)
effect_df_P07

  [P07] UFM1 细胞数不足或不在wPSI矩阵里，跳过
  [P07] SRPRB 细胞数不足或不在wPSI矩阵里，跳过
  [P07] MESD 细胞数不足或不在wPSI矩阵里，跳过
  [P07] CD38 细胞数不足或不在wPSI矩阵里，跳过
  [P07] leading gene组内对照分类: supports=2, opposes=1, no_meaningful_diff=3


,gene_id,nonzero_frac_a,nonzero_frac_b,mean_a,mean_b,pval,mean_diff,abs_mean_diff,category
symbol,,,,,,,,,
TMBIM6,ENSG00000139644.13,0.288462,0.636364,0.042328,0.210222,8.831803e-04,0.167894,0.167894,supports
SEC61A1,ENSG00000058262.11,1.000000,0.850000,0.403718,0.263234,7.657317e-08,-0.140484,0.140484,opposes
COPZ1,ENSG00000111481.10,1.000000,1.000000,0.255745,0.371938,1.991147e-03,0.116193,0.116193,supports
SPCS3,ENSG00000129128.13,0.771930,0.791667,0.178815,0.127640,8.840533e-01,-0.051176,0.051176,no_meaningful_diff
KDELR2,ENSG00000136240.10,0.957447,1.000000,0.818554,0.862465,9.249863e-01,0.043911,0.043911,no_meaningful_diff
SPCS2,ENSG00000118363.13,0.818182,0.925926,0.203318,0.216720,7.848967e-01,0.013401,0.013401,no_meaningful_diff


In [26]:
# %% [Layer 1 深入] P07 画图
plot_within_celltype_effect_summary(
    effect_df_P07, "P07", "Plasma cells",
    CONFIG["output_root"] + "layer1",
)

  已保存: /Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_within_celltype_effect_P07.png


'/Volumes/Elements/DEPA_downstream/outputs/pre_T/layer1/layer1_within_celltype_effect_P07.png'

In [27]:
# %% [Layer 2] program usage 与各 contrast（疾病、细胞身份）的关联
assoc_results = compute_contrast_association(data, CONFIG, program_direction=program_direction)
assoc_results["disease"]


  contrast 'celltype_identity' (auto_one_vs_rest_all, column='cell_type_final') 自动展开为 10 条 [category=identity]: ['Erythroid cells', 'HSPC', 'Monocytes', 'NK cells', 'Naive B cells', 'Normal T cells', 'Plasma cells', 'Proliferating T-ALL', 'T-ALL cells', 'pDC cells']

=== contrast 'disease' [disease] (column='group', type='pairwise') ===
  PBMMC: 9161 细胞  |  PreT_ALL: 8226 细胞
  最强关联: P02 (|AUC-0.5|=0.398)

=== contrast 'Erythroid cells' [identity] (column='cell_type_final', type='one_vs_rest') ===
  not_Erythroid cells: 14650 细胞  |  Erythroid cells: 2737 细胞
  最强关联: P06 (|AUC-0.5|=0.295)

=== contrast 'HSPC' [identity] (column='cell_type_final', type='one_vs_rest') ===
  not_HSPC: 17070 细胞  |  HSPC: 317 细胞
  最强关联: P05 (|AUC-0.5|=0.395)

=== contrast 'Monocytes' [identity] (column='cell_type_final', type='one_vs_rest') ===
  not_Monocytes: 15988 细胞  |  Monocytes: 1399 细胞
  最强关联: P04 (|AUC-0.5|=0.489)

=== contrast 'NK cells' [identity] (column='cell_type_final', type='one_vs_rest') ===
  

,auc,lr_coef,dt_importance,xgb_importance,n_group_a,n_group_b,direction
program,,,,,,,
P02,0.897755,3.602907,0.610609,0.455825,9161,8226,Proximal
P03,0.217135,-1.587531,0.065625,0.078839,9161,8226,Proximal
P01,0.740529,0.850221,0.125642,0.112925,9161,8226,Proximal
P06,0.279688,-2.061735,0.000000,0.058636,9161,8226,Proximal
P05,0.307585,-2.481527,0.198124,0.202565,9161,8226,Proximal
P04,0.330784,-0.579420,0.000000,0.027718,9161,8226,Distal
P08,0.582521,0.497256,0.000000,0.050620,9161,8226,Proximal
P07,0.557143,0.049245,0.000000,0.012874,9161,8226,Distal


In [28]:
# %% [Layer 2] 关联结果可视化：热图 + UMAP网格
plot_association_heatmap(assoc_results, CONFIG, CONFIG["output_root"] + "layer2")
plot_program_umap_grid(data, CONFIG, CONFIG["output_root"] + "layer2")


  已保存: /Volumes/Elements/DEPA_downstream/outputs/pre_T/layer2/layer2_association_heatmap.png
  已保存: /Volumes/Elements/DEPA_downstream/outputs/pre_T/layer2/layer2_program_umap_grid.png


'/Volumes/Elements/DEPA_downstream/outputs/pre_T/layer2/layer2_program_umap_grid.png'

In [29]:
# %% [Layer 2 深入] P04 program usage 组内对照（Monocytes内部，排除细胞组成混淆）
compare_within_celltype(
    data, CONFIG,
    celltype="Monocytes",
    disease_column=CONFIG["contrasts"][0]["column"],
    group_a="PBMMC", group_b="PreT_ALL",
    program="P04",
)

=== 组内比较: celltype='Monocytes', group: PBMMC vs PreT_ALL ===
  该celltype总细胞数: 1399 (PBMMC: 1156, PreT_ALL: 243)
  [usage:P04] PBMMC median=0.7011 (n=1156)  PreT_ALL median=0.5678 (n=243)  pval=1.817e-11


{'usage': {'n_a': 1156,
  'n_b': 243,
  'median_a': 0.70114777,
  'median_b': 0.5678302,
  'pval': 1.8166453535302448e-11}}

In [30]:
# %% [Layer 2 深入] P07 program usage 组内对照（Plasma cells内部）
compare_within_celltype(
    data, CONFIG,
    celltype="Plasma cells",
    disease_column=CONFIG["contrasts"][0]["column"],
    group_a="PBMMC", group_b="PreT_ALL",
    program="P07",
)

=== 组内比较: celltype='Plasma cells', group: PBMMC vs PreT_ALL ===
  该celltype总细胞数: 115 (PBMMC: 84, PreT_ALL: 31)
  [usage:P07] PBMMC median=0.5481 (n=84)  PreT_ALL median=0.6467 (n=31)  pval=4.630e-04


{'usage': {'n_a': 84,
  'n_b': 31,
  'median_a': 0.548091505,
  'median_b': 0.6467378,
  'pval': 0.00046304007792018795}}

In [31]:
# %% [Layer 3] program核心基因集 / Jaccard相似度 / 专属marker
program_gene_sets = compute_program_gene_sets(data, CONFIG)
jaccard_mat = compute_jaccard_matrix(program_gene_sets["gene_sets"], data["programs"])
markers = select_program_markers(program_gene_sets, data, CONFIG, top_n=20, spec_min_z=0.5)

  [P01] 核心基因集大小: 27
  [P02] 核心基因集大小: 36
  [P03] 核心基因集大小: 24
  [P04] 核心基因集大小: 28
  [P05] 核心基因集大小: 36
  [P06] 核心基因集大小: 15
  [P07] 核心基因集大小: 19
  [P08] 核心基因集大小: 18
  [P01] 候选基因27个 -> 专属marker8个 (spec_min_z=0.5)
  [P02] 候选基因36个 -> 专属marker8个 (spec_min_z=0.5)
  [P03] 候选基因24个 -> 专属marker9个 (spec_min_z=0.5)
  [P04] 候选基因28个 -> 专属marker15个 (spec_min_z=0.5)
  [P05] 候选基因36个 -> 专属marker1个 (spec_min_z=0.5)
  [P06] 候选基因15个 -> 专属marker2个 (spec_min_z=0.5)
  [P07] 候选基因19个 -> 专属marker6个 (spec_min_z=0.5)
  [P08] 候选基因18个 -> 专属marker2个 (spec_min_z=0.5)


In [32]:
# %% [Layer 3] 模块标签推断 + Jaccard热图
module_labels = resolve_module_labels(data, CONFIG)
plot_jaccard_heatmap(jaccard_mat, CONFIG["output_root"] + "layer3", module_labels=module_labels)


  [Layer3] depa_feature_source 没有单值的'context_id'列，使用'contexts'列（可能一个PAC对应多个context，将展开逐个计数）
  contexts列原始样例: ['PreT_ALL_Normal_T_cells_vs_PBMMC_Normal_T_cells', 'PreT_ALL_Erythroid_cells_vs_PBMMC_Erythroid_cells;PreT_ALL_Naive_B_cells_vs_PBMMC_Naive_B_cells;PreT_ALL_NK_cells_vs_PBMMC_NK_cells;PreT_ALL_Normal_T_cells_vs_PBMMC_Normal_T_cells;PreT_ALL_Proliferating_T-ALL_vs_PBMMC_Proliferating_T-ALL', 'PreT_ALL_Erythroid_cells_vs_PBMMC_Erythroid_cells;PreT_ALL_Naive_B_cells_vs_PBMMC_Naive_B_cells;PreT_ALL_Normal_T_cells_vs_PBMMC_Normal_T_cells;PreT_ALL_Proliferating_T-ALL_vs_PBMMC_Proliferating_T-ALL;PreT_ALL_vs_PBMMC']
  [P01] top_contexts: [('PreT_ALL_NK_cells_vs_PBMMC_NK_cells', np.float64(0.154)), ('PreT_ALL_Proliferating_T-ALL_vs_PBMMC_Proliferating_T-ALL', np.float64(0.135)), ('PreT_ALL_Normal_T_cells_vs_PBMMC_Normal_T_cells', np.float64(0.096))]
  [P02] top_contexts: [('Other_vs_Normal_T_cells', np.float64(0.171)), ('PreT_ALL_Proliferating_T-ALL_vs_PBMMC_Proliferating_T-ALL', np.fl

'/Volumes/Elements/DEPA_downstream/outputs/pre_T/layer3/layer3_jaccard_heatmap.png'

In [41]:
# %% [Layer 3] 专属marker的功能富集
gene_lists = {f"{p}_markers": markers[p]["symbol"].tolist() for p in data["programs"]}
enrichment = run_enrichment(gene_lists, CONFIG)
plot_enrichment_bubble(enrichment, CONFIG["output_root"] + "layer3", top_n=6)


  [P01_markers] 过滤掉 1 个仍是ENSG格式(未成功映射到symbol)的基因，剩余 7 个送去Enrichr查询
  [P01_markers] 7个基因 -> 126条富集term，最强: Spliceosomal Complex Assembly (GO:0000245) (padj=1.24e-02)
  [P02_markers] 8个基因 -> 153条富集term，最强: G2-M Checkpoint (padj=2.66e-04)
  [P03_markers] 9个基因 -> 142条富集term，最强: 7-Methylguanosine RNA Capping (GO:0009452) (padj=4.47e-02)
  [P04_markers] 15个基因 -> 356条富集term，最强: Immune System R-HSA-168256 (padj=1.03e-02)
  [P05_markers] 基因数 1 < min_genes=3，跳过
  [P06_markers] 基因数 2 < min_genes=3，跳过
  [P07_markers] 6个基因 -> 68条富集term，最强: Protein export (padj=1.33e-07)
  [P08_markers] 基因数 2 < min_genes=3，跳过
  已保存: /Volumes/Elements/DEPA_downstream/outputs/pre_T/layer3/layer3_enrichment_bubble.png


'/Volumes/Elements/DEPA_downstream/outputs/pre_T/layer3/layer3_enrichment_bubble.png'

In [34]:
# %% [Layer 3] 全部核心基因（非仅专属marker）的功能富集，作为对照
symbol_map = data.get("gene_symbol_map")

def _symbol(gene_id):
    if symbol_map is None:
        return gene_id
    if gene_id in symbol_map.index:
        return symbol_map.loc[gene_id]
    bare_id = gene_id.split(".")[0]
    if bare_id in symbol_map.index:
        return symbol_map.loc[bare_id]
    return gene_id

gene_lists_all = {}
for p in data["programs"]:
    ensg_list = program_gene_sets["gene_sets"][p]
    symbols = [_symbol(g) for g in ensg_list]
    n_before = len(symbols)
    symbols = [s for s in symbols if not str(s).startswith("ENSG")]
    gene_lists_all[f"{p}_all"] = symbols
    print(f"{p}: 核心基因{n_before}个 -> 映射到symbol {len(symbols)}个")

enrichment_all = run_enrichment(gene_lists_all, CONFIG)

plot_enrichment_bubble(
    enrichment_all,
    CONFIG["output_root"] + "layer3",
    top_n=6,
    filename="layer3_enrichment_bubble_all.png",
)

P01: 核心基因27个 -> 映射到symbol 26个
P02: 核心基因36个 -> 映射到symbol 35个
P03: 核心基因24个 -> 映射到symbol 24个
P04: 核心基因28个 -> 映射到symbol 28个
P05: 核心基因36个 -> 映射到symbol 36个
P06: 核心基因15个 -> 映射到symbol 15个
P07: 核心基因19个 -> 映射到symbol 19个
P08: 核心基因18个 -> 映射到symbol 18个
  [P01_all] 26个基因 -> 482条富集term，最强: L13a-mediated Translational Silencing Of Ceruloplasmin Expression R-HSA-156827 (padj=7.88e-15)
  [P02_all] 35个基因 -> 858条富集term，最强: E2F Targets (padj=1.65e-11)
  [P03_all] 24个基因 -> 598条富集term，最强: KRAS Signaling Up (padj=1.67e-03)
  [P04_all] 28个基因 -> 793条富集term，最强: Immune System R-HSA-168256 (padj=1.33e-06)
  [P05_all] 36个基因 -> 934条富集term，最强: E2F Targets (padj=2.26e-11)
  [P06_all] 15个基因 -> 343条富集term，最强: heme Metabolism (padj=3.17e-21)
  [P07_all] 19个基因 -> 268条富集term，最强: Protein processing in endoplasmic reticulum (padj=2.18e-09)
  [P08_all] 18个基因 -> 436条富集term，最强: heme Metabolism (padj=8.83e-22)
  已保存: /Volumes/Elements/DEPA_downstream/outputs/pre_T/layer3/layer3_enrichment_bubble_all.png


'/Volumes/Elements/DEPA_downstream/outputs/pre_T/layer3/layer3_enrichment_bubble_all.png'

In [43]:
# 接在 Part 3 之后，复用已有的 run_enrichment
supports_genes_P02 = effect_df_P02[effect_df_P02["category"] == "supports"].index.tolist()
opposes_genes_P02 = effect_df_P02[effect_df_P02["category"] == "opposes"].index.tolist()

gene_lists_split = {
    "P02_supports": supports_genes_P02,
    "P02_opposes": opposes_genes_P02,
}
enrichment_split = run_enrichment(gene_lists_split, CONFIG, min_genes=2)  # opposes只有2个基因，min_genes要调低
plot_enrichment_bubble(enrichment_split, CONFIG["output_root"] + "layer3", top_n=8, filename="layer3_enrichment_split_P02.png")

  [P02_supports] 6个基因 -> 243条富集term，最强: E2F Targets (padj=3.85e-05)
  [P02_opposes] 2个基因 -> 19条富集term，最强: Wnt signaling pathway (padj=1.65e-02)
  已保存: /Volumes/Elements/DEPA_downstream/outputs/pre_T/layer3/layer3_enrichment_split_P02.png


'/Volumes/Elements/DEPA_downstream/outputs/pre_T/layer3/layer3_enrichment_split_P02.png'

In [39]:
supports_set_P02 = set(effect_df_P02[effect_df_P02["category"] == "supports"].index)
marker_symbols_P02 = set(markers["P02"]["symbol"])
overlap = supports_set_P02 & marker_symbols_P02
print(f"P02 supports基因: {supports_set_P02}")
print(f"P02 Layer3专属marker: {marker_symbols_P02}")
print(f"两者重合: {overlap} ({len(overlap)}/{len(supports_set_P02)})")

P02 supports基因: {'DNTT', 'CENPM', 'NAP1L1', 'CARHSP1', 'RAN', 'PSMB2'}
P02 Layer3专属marker: {'JPT1', 'MPC1', 'PTP4A2', 'MKI67', 'MZT1', 'CALM3', 'HNRNPA0', 'TPX2'}
两者重合: set() (0/6)


In [40]:
gene_lists_retry = {"P02_supports": supports_genes_P02}
enrichment_supports_retry = run_enrichment(gene_lists_retry, CONFIG, min_genes=2)

  [P02_supports] 6个基因 -> 243条富集term，最强: E2F Targets (padj=3.85e-05)


In [42]:
print(enrichment_split.keys())
print(len(enrichment_split.get("P02_supports", [])) if enrichment_split.get("P02_supports") is not None else "P02_supports缺失或为None")

dict_keys(['P02_supports', 'P02_opposes'])
243
